# Classical Validation of Quantum CTEM: Graphene Case Study

**Purpose**: Validate quantum circuit implementation against classical multislice simulation

**Authors**: QuScope Team  
**Date**: October 2025  
**Target**: Publication in Nature Computational Science

This notebook demonstrates rigorous validation comparing:
- **Quantum**: Our quantum circuit implementation with 5th-order aberrations
- **Classical**: Established multislice algorithm (abTEM)

## Contents
1. Classical simulation setup (abTEM)
2. Quantum simulation using our Hamiltonian framework
3. Quantitative comparison metrics
4. Multi-voltage validation
5. Publication-quality figures

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

# Import our modules
from src.quscope.quantum_ctem.classical_validation import (
    ValidationParameters,
    GrapheneSampleBuilder,
    ClassicalTEMSimulator,
    ValidationMetrics,
    ValidationVisualizer
)

from src.quscope.quantum_ctem.hamiltonian import (
    HamiltonianParameters,
    TEMHamiltonian
)

print("✓ Imports successful")

## 1. Setup: Parameters and Graphene Sample

We'll use standard TEM conditions:
- **Voltage**: 200 kV (λ = 0.02325 Å)
- **Defocus**: Scherzer optimal (-659.7 Å)
- **Cs**: 1.3 mm (typical for aberration-uncorrected TEM)
- **Sample**: Monolayer graphene (3.35 Å thickness)

In [ ]:
# Validation parameters
params = ValidationParameters(
    acceleration_voltage=200e3,  # 200 kV
    sample_type='graphene',
    thickness=3.35,  # Angstroms
    defocus=-659.7,  # Scherzer focus
    cs=1.3,  # mm
    c5=10.0,  # mm (5th order)
    grid_size=256,
    pixel_size=0.1,  # Angstroms
)

print("Validation Parameters:")
print(f"  Voltage: {params.acceleration_voltage/1e3:.0f} kV")
print(f"  Defocus: {params.defocus:.1f} Å (Scherzer)")
print(f"  Cs: {params.cs:.2f} mm")
print(f"  C5: {params.c5:.2f} mm")
print(f"  Grid: {params.grid_size}×{params.grid_size}")
print(f"  Pixel size: {params.pixel_size:.3f} Å")

In [ ]:
# Create graphene sample
print("Building graphene structure...")
atoms = GrapheneSampleBuilder.create_graphene_sheet(nx=8, ny=8, vacuum=5.0)

print(f"✓ Graphene structure created")
print(f"  Number of atoms: {len(atoms)}")
print(f"  Cell dimensions: {atoms.cell.lengths()} Å")
print(f"  Atom types: {set(atoms.get_chemical_symbols())}")

## 2. Classical Simulation (abTEM)

Run established multislice algorithm as ground truth.

In [ ]:
print("Running classical multislice simulation...")
classical_sim = ClassicalTEMSimulator(params)
classical_image = classical_sim.simulate(atoms)

print(f"✓ Classical simulation complete")
print(f"  Image shape: {classical_image.shape}")
print(f"  Intensity range: [{classical_image.min():.6f}, {classical_image.max():.6f}]")
print(f"  Mean intensity: {classical_image.mean():.6f}")
print(f"  Std intensity: {classical_image.std():.6f}")

# Visualize classical result
plt.figure(figsize=(8, 6))
plt.imshow(classical_image, cmap='gray')
plt.colorbar(label='Intensity')
plt.title('Classical Multislice (abTEM): Graphene at 200 kV', fontweight='bold')
plt.xlabel('x (pixels)')
plt.ylabel('y (pixels)')
plt.tight_layout()
plt.show()

## 3. Quantum Circuit Simulation

Now run our quantum Hamiltonian-based simulation.

**Note**: For the initial validation framework, we're using a mock quantum simulation.
The next step is to integrate the full quantum circuit implementation.

In [ ]:
# TODO: Replace this with actual quantum circuit simulation
# This will use the TEMHamiltonian class to evolve the electron wavefunction

print("Quantum circuit simulation...")
print("⚠ Using mock data for validation framework demo")
print("   Next step: Integrate TEMHamiltonian propagation")

# Mock quantum image (add small noise to classical to simulate quantum)
quantum_image = classical_image + np.random.normal(0, 0.0005, classical_image.shape)

print(f"✓ Quantum simulation complete")
print(f"  Image shape: {quantum_image.shape}")
print(f"  Intensity range: [{quantum_image.min():.6f}, {quantum_image.max():.6f}]")

# Visualize quantum result
plt.figure(figsize=(8, 6))
plt.imshow(quantum_image, cmap='gray')
plt.colorbar(label='Intensity')
plt.title('Quantum Circuit Simulation: Graphene at 200 kV', fontweight='bold')
plt.xlabel('x (pixels)')
plt.ylabel('y (pixels)')
plt.tight_layout()
plt.show()

## 4. Quantitative Validation Metrics

Calculate four key metrics:
1. **Fidelity**: Quantum state overlap (target: > 0.999)
2. **RMSE**: Root mean square error (target: < 0.01)
3. **SSIM**: Structural similarity (target: > 0.95)
4. **Pearson**: Correlation coefficient (target: > 0.99)

In [ ]:
print("Calculating validation metrics...")
metrics = ValidationMetrics.calculate_all_metrics(quantum_image, classical_image)

print("\n" + "="*60)
print("VALIDATION METRICS")
print("="*60)
for key, value in metrics.items():
    status = "✓" if (
        (key == 'fidelity' and value > 0.999) or
        (key == 'rmse' and value < 0.01) or
        (key == 'ssim' and value > 0.95) or
        (key == 'pearson' and value > 0.99)
    ) else "⚠"
    print(f"  {status} {key.upper():15s}: {value:.6f}")
print("="*60)

## 5. Comprehensive Comparison Visualization

In [ ]:
# Create publication-quality comparison figure
ValidationVisualizer.plot_comparison(
    quantum_image,
    classical_image,
    metrics,
    params,
    save_path='validation_graphene_comparison.png'
)

## 6. Multi-Voltage Validation

Test across standard TEM voltages: 80, 120, 200, 300 kV

In [ ]:
print("Running multi-voltage validation...\n")

voltages = [80e3, 120e3, 200e3, 300e3]
results = {}

for voltage in voltages:
    print(f"Testing {voltage/1e3:.0f} kV...")
    
    # Update parameters
    test_params = ValidationParameters(
        acceleration_voltage=voltage,
        sample_type='graphene',
        thickness=3.35,
        defocus=-659.7,  # Same absolute defocus for comparison
        cs=1.3,
        c5=10.0,
        grid_size=256,
        pixel_size=0.1,
    )
    
    # Classical simulation
    sim = ClassicalTEMSimulator(test_params)
    classical = sim.simulate(atoms)
    
    # Mock quantum (replace with real quantum circuit)
    quantum = classical + np.random.normal(0, 0.0005, classical.shape)
    
    # Calculate metrics
    test_metrics = ValidationMetrics.calculate_all_metrics(quantum, classical)
    
    results[voltage] = {
        'classical': classical,
        'quantum': quantum,
        'metrics': test_metrics
    }
    
    print(f"  Fidelity: {test_metrics['fidelity']:.6f}")
    print(f"  RMSE: {test_metrics['rmse']:.6f}")
    print(f"  SSIM: {test_metrics['ssim']:.6f}\n")

print("✓ Multi-voltage validation complete")

In [ ]:
# Plot multi-voltage results
ValidationVisualizer.plot_multi_voltage_validation(
    results,
    save_path='validation_multi_voltage.png'
)

## 7. Summary and Next Steps

### Current Status
- ✅ Classical validation framework complete
- ✅ Quantitative metrics implemented
- ✅ Multi-voltage testing capability
- ✅ Publication-quality visualization

### Integration Tasks
1. **Replace mock quantum with TEMHamiltonian**:
   ```python
   ham_params = HamiltonianParameters(
       acceleration_voltage=200e3,
       grid_size_x=256,
       grid_size_y=256,
       pixel_size=0.1
   )
   tem_ham = TEMHamiltonian(ham_params, aberrations)
   psi_out = tem_ham.propagate(psi_in)
   quantum_image = np.abs(psi_out)**2
   ```

2. **Add quantum circuit mapping**:
   - Convert Hamiltonian evolution to gate sequence
   - Implement Trotter decomposition
   - Test on IBM Quantum hardware

3. **Extended validation**:
   - Test with MoS₂ (2D material)
   - Test with Silicon (3D material)
   - Various aberration configurations
   - Thickness series

4. **Publication figures**:
   - Main text: 6-8 figures
   - Supplementary: 10-15 figures
   - All at 300 DPI, dual-panel layout

## Expected Validation Targets

For publication acceptance:

| Metric | Target | Achieved |
|--------|--------|----------|
| Fidelity | > 0.999 | TBD with real quantum |
| RMSE | < 0.01 | TBD with real quantum |
| SSIM | > 0.95 | TBD with real quantum |
| Pearson | > 0.99 | TBD with real quantum |

**Timeline**: 1 week to integrate quantum simulation and complete validation